[Guide Index](README.md)


# 05.01 Validating with SHACL 1.2

`starlayer.shacl` is a SHACL engine written against the SHACL 1.2 draft specifications.

This guide covers validating data with SHACL 1.2. It assumes a working knowledge of SHACL 1.1 and covers what has been added by SHACL 1.2 and the functionality of `starlayer.shacl`.  Basic SHACL 1.1 functionality is not covered.

The guide starts with the standard way to validate - a `validate()` function. It then introduces the `StarShaclProcessor` class, which is unique to StarLayer: it binds a shapes graph to various processing modes so it can be re-used with different input graphs. The guide then covers what SHACL 1.2 adds to validation (over SHACL 1.1), and how to read a result. 

SHACL 1.2 implements many of the features of the SHACL Advanced Features Note (SHACL-AF). Rules (`sh:rule`, `sh:TripleRule`, `sh:SPARQLRule`, `sh:condition`) are now defined in the Inference Rules draft, and AF's expressions are replaced by the `shnex:` Node Expressions. `starlayer.shacl` implements SHACL 1.2. SHACL-AF features that are not included in SHACL 1.2 are not supported. Shapes written using SHACL-AF may be invalid or return unexpected results.  

Additional guides are provided to explore more in-depth SHACL 1.2 functionality supported by StarLayer.

- **[SHACL node expressions](05.04-shacl-node-expressions.ipynb)** — the `shnex:`/`sparql:` node-expression vocabulary.
- **[SHACL and SPARQL](05.05-shacl-and-sparql.ipynb)** — `sh:sparql` constraints, user-defined `sh:ConstraintComponent`.
- **[SHACL inference rules](05.06-shacl-inference-rules.ipynb)** — `sh:rule`/`sh:TripleRule`/`sh:SPARQLRule` rule execution.
- **[SHACL UI](05.07-shacl-ui.ipynb)** — `shui:` widgets, widget selection and scoring.
- **[SHACL profiling](05.08-shacl-profiling.ipynb)** — `sh:ShapesGraph`/`sh:DataGraph`/`owl:imports` packaging conventions, profiling with SHACL and profiling of SHACL.
- **[SHACL subgraph extraction](05.09-shacl-subgraph-extraction.ipynb)** — `extract_subgraph()`, `close_shape()`, and the hash-and-verify commitment pattern.
- **[Building on SHACL](05.10-building-on-shacl.ipynb)** — the functions an application that edits or displays shapes uses.
- **[SHACL evaluate()](05.11-shacl-evaluate.ipynb)** — selecting the nodes and values to present, and computing values and defaults.
- **[The StarLayer SHACL engine (05.02)](05.02-starlayer-shacl-engine.ipynb)** — the processor and the functions of `starlayer.shacl`: what is a method and what is only a function, configuring the processor, deriving shapes from data, checking a shapes graph, the graphs the engine accepts, and the properties that describe a shape without validating.
- **[SHACL shapes as classes (05.03)](05.03-shapes-as-classes.ipynb)** — shapes that are also classes, inheriting properties through the class hierarchy, and reading the hierarchy from the shapes graph.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier ones.


In [1]:
from starlayer.graph import Namespace
from starlayer.graph import StarLayerGraph
from starlayer.graph.config import set_default_prefixes
from starlayer.shacl import StarShaclProcessor, target_nodes, validate

# EX and SH are starlayer.graph Namespaces: their terms are pyoxigraph NamedNodes, used to look terms up in the results below
# (StarLayerGraphs, see section 05.01.2); the StarLayerGraph inputs are built by parsing Turtle.
EX = Namespace("http://example.org/")
SH = Namespace("http://www.w3.org/ns/shacl#")

# Every StarLayerGraph created after this uses these prefixes, so results, queries and the report text use short names such as
# ex:alice 
set_default_prefixes({
    "ex": "http://example.org/",
    "sh": "http://www.w3.org/ns/shacl#",
    "shnex": "http://www.w3.org/ns/shacl-node-expr#",
    "prov": "http://www.w3.org/ns/prov#",
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "xsd": "http://www.w3.org/2001/XMLSchema#",
})


def show(graph):
    """Print a graph as pretty Turtle, declaring only the prefixes it uses."""
    print(graph.serialize(format="turtle", pretty=True, only_used=True))

## 05.01.1 Standard SHACL validation

SHACL validation takes a data graph and a shapes graph and reports whether the data conforms to the shapes. `validate()` operates in a form familiar from other SHACL libraries (for example the `pyshacl` package): pass the data graph and the shapes graph, and get a result in response. The result has `conforms` (boolean), the report as a graph (`report_graph`) and the report as text (`report_text`).

Below, the shape says every `ex:Person` needs at least one `ex:name`. `ex:alice` has one (conforms) and `ex:bob` has none (does not conform).

In [2]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ] .
""", format="turtle")

people = StarLayerGraph()
people.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:name "Alice" .
    ex:bob a ex:Person .
""", format="turtle")

result = validate(people, shacl_graph=shapes)
print("conforms:", result.conforms)
print(result.report_text)

conforms: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in sh:MinCountConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:minCount 1 ; sh:path ex:name ]
	Focus Node: ex:bob
	Result Path: ex:name
	Message: Needs at least 1 value for ex:name



## 05.01.2 The `StarShaclProcessor` class

`StarShaclProcessor` is `starlayer.shacl`'s own SHACL entry point. It is created from a shapes graph, which it then retains and can use with multiple data graphs. The processor can also hold settings that apply to every call, such as `allow_warnings` or `inference`. They are described in [the engine guide](05.02-starlayer-shacl-engine.ipynb).

The processor has three independent processing modes over the same shapes graph and data graph:

- **`validate()`** checks whether the data conforms to the shapes, and never changes it. It is the subject of this guide.
- **`evaluate()`** selects the nodes and property values to present. It is covered in the [SHACL evaluate guide](05.11-shacl-evaluate.ipynb).
- **`apply_rules()`** runs the `sh:rule` rules of the shapes graph to a fixed point to create a new graph. It is covered in the [SHACL inference rules guide](05.06-shacl-inference-rules.ipynb).

`StarShaclProcessor` can use a `StarLayerGraph` or a `StarLayerDataset` for the data and the shapes graphs. A plain `pyoxigraph.Store` also works. Graph-based results are returned as `StarLayerGraph`s (e.g. `result.report_graph`).

Below, a processor is created from the `shapes` graph of section 05.01.1 and used to validate two data graphs: `people`, where `ex:bob` has no name (does not conform), and a corrected graph (conforms). This is the same check as the `validate()` function.

In [3]:
# the processor is created from the shapes graph, then used to validate a data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)
result = myProcessor.validate(data_graph=people)
print("conforms:", result.conforms)
print(result.report_text)

# the same processor validates another data graph
fixed = StarLayerGraph()
fixed.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:bob a ex:Person ; ex:name "Bob" .
""", format="turtle")
print("conforms:", myProcessor.validate(data_graph=fixed).conforms)


conforms: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in sh:MinCountConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:minCount 1 ; sh:path ex:name ]
	Focus Node: ex:bob
	Result Path: ex:name
	Message: Needs at least 1 value for ex:name

conforms: True


## 05.01.3 `validate()` — conformance checking

`StarShaclProcessor.validate()` is the same check as the `validate()` function, provided as a method of the processor.  Given a data graph, report whether it conforms to the shapes graph the processor holds. This section covers what SHACL 1.2 adds to validation, one area at a time:

- **05.01.3.1 Working with RDF 1.2 terms**: triple terms, reifiers and direction-tagged strings.
- **05.01.3.2 New constraint components**: `sh:singleLine`, `sh:someValue` and `sh:uniqueValuesFor`.
- **05.01.3.3 Lists**: constraints on the members of an RDF list, and lists of classes and datatypes.
- **05.01.3.4 Property path comparisons**: `sh:equals`, `sh:lessThan` and similar expressions. 
- **05.01.3.5 Values below a root class**: `sh:rootClass`. Shapes that are also classes have a [guide of their own](05.03-shapes-as-classes.ipynb).
- **05.01.3.6 Targets**: `sh:targetWhere` and `sh:shape`.
- **05.01.3.7 Severity, messages and conformance**: the new severities, fine-grained severity, and which severities make a report fail.
- **05.01.3.8 Constant value nodes**: `sh:defaultValue` and `sh:values`.

### 05.01.3.1 Working with RDF 1.2 terms

RDF 1.2 adds triple terms, reifiers and direction-tagged strings. SHACL 1.2 can check each of them.

#### 05.01.3.1.1 Is a value a triple term? `sh:nodeKind ( sh:TripleTerm )`

SHACL 1.2 allows `sh:nodeKind` to check that a node is a `sh:TripleTerm`.  This allows a shape to require that a value (e.g. the object of `rdf:reifies`) is an RDF 1.2 triple term.

Below, a data graph whose `rdf:reifies` object is a triple term (conforms) and one whose object is a plain IRI (does not conform).

In [4]:
# reifiers should only reify triple terms.
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    [] a sh:NodeShape ;
      sh:targetSubjectsOf rdf:reifies ;
      sh:property [ sh:path rdf:reifies ; sh:nodeKind sh:TripleTerm  ] .
""", format="turtle")

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .
""", format="turtle")

# processor is created from the shapes graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)

# validation is run on data graph
result = myProcessor.validate(data_graph=data)
print("conforms (real triple term):", result.conforms)
print(result.report_text)

bad_data = StarLayerGraph()
bad_data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:claim rdf:reifies ex:not_a_triple_term .
""", format="turtle")

# validation is run on different data graph using same processor.
result_bad = myProcessor.validate(data_graph=bad_data)
print("conforms (plain URI, not a triple term):", result_bad.conforms)
print(result_bad.report_text)

conforms (real triple term): True
Validation Report
Conforms: True

conforms (plain URI, not a triple term): False
Validation Report
Conforms: False
Results (1):
Constraint Violation in sh:NodeKindConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:nodeKind sh:TripleTerm ; sh:path rdf:reifies ]
	Focus Node: ex:claim
	Value Node: ex:not_a_triple_term
	Result Path: rdf:reifies
	Message: Value ex:not_a_triple_term should be a triple term



#### 05.01.3.1.2 Reifiers of a value: `sh:reifierShape` and `sh:reificationRequired`

In RDF 1.2 a triple can have a reifier, a node linked to the triple by `rdf:reifies`, which carries facts about the claim itself (who made it, when). 

On a property shape, `sh:reifierShape` names a node shape that every reifier of a matching triple must conform to. For each value node, the triple (focus node, path, value) is looked up and each of its reifiers is validated against that shape. 

`sh:reificationRequired true` requires each such triple to have at least one reifier. The path must be a plain predicate IRI.

Below, the age claim of `ex:Alice` has a reifier with a date and an author (conforms), the claim of `ex:Bob` has a reifier without an author (does not conform), and the claim of `ex:Carol` has no reifier (does not conform).

In [5]:
# age triples require a reification and that reification must comply with the ClaimShape (at least one date and one author)
shapes = StarLayerGraph()
shapes.parse(format="turtle", data="""
@prefix ex: <http://example.org/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person ;
  sh:property [ sh:path ex:age ; sh:reifierShape ex:ClaimShape ; sh:reificationRequired true ] .
ex:ClaimShape a sh:NodeShape ;
  sh:property [ sh:path ex:date ; sh:minCount 1 ] ;
  sh:property [ sh:path ex:author ; sh:minCount 1 ] .
""")

# the processor is created from the shapes graph, then used to validate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)

# the data uses RDF 1.2 syntax, so {| ... |} is the reifier of the triple before it
alice = StarLayerGraph()
alice.parse(format="turtle", data="""
    @prefix ex: <http://example.org/> .
    ex:Alice a ex:Person ; ex:age 23 {| ex:date '2019-12-05' ; ex:author ex:Claire |} .
""")
bob = StarLayerGraph()
bob.parse(format="turtle", data="""
    @prefix ex: <http://example.org/> .
    ex:Bob a ex:Person ; ex:age 23 {| ex:date '2019-12-05' |} .
""")
carol = StarLayerGraph()
carol.parse(format="turtle", data="""
    @prefix ex: <http://example.org/> .
    ex:Carol a ex:Person ; ex:age 23 .
""")

for data in (alice, bob, carol):
    person = next(data.subjects(EX.age, None))
    result = myProcessor.validate(data_graph=data)
    print(data.qname(person), "conforms:", result.conforms)
    print(result.report_text)

ex:Alice conforms: True
Validation Report
Conforms: True

ex:Bob conforms: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in sh:ReifierShapeConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:path ex:age ; sh:reificationRequired true ; sh:reifierShape ex:ClaimShape ]
	Focus Node: ex:Bob
	Value Node: 23
	Result Path: ex:age
	Message: The claim (ex:Bob ex:age 23) has a reifier that does not conform to ex:ClaimShape

ex:Carol conforms: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in sh:ReifierShapeConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:path ex:age ; sh:reificationRequired true ; sh:reifierShape ex:ClaimShape ]
	Focus Node: ex:Carol
	Value Node: 23
	Result Path: ex:age
	Message: The claim (ex:Carol ex:age 23) has no reifier



#### 05.01.3.1.3 Direction-aware datatype constraints

The RDF 1.2 `rdf:dirLangString` (the datatype of a direction-tagged literal such as `"hello"@en--ltr`) works as an ordinary `sh:datatype` value in SHACL 1.2.

Below, `ex:n1` has one label with a direction and one label without one, and the label without a direction does not conform.

In [6]:
# two labels.  one is not direction tagged literal.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:n1 a ex:Note ;
      ex:label "hello"@en--ltr ;
      ex:label "hello"@fr .
""", format="turtle")

# all labels need direction
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:LangShape a sh:NodeShape ;
      sh:targetClass ex:Note ;
      sh:property [ sh:path ex:label ; sh:datatype rdf:dirLangString ] .
""", format="turtle")
# the processor is created from the shapes graph, then used to validate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)
result = myProcessor.validate(data_graph=data)
print("conforms:", result.conforms)
print(result.report_text)

conforms: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in sh:DatatypeConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:datatype rdf:dirLangString ; sh:path ex:label ]
	Focus Node: ex:n1
	Value Node: "hello"@fr
	Result Path: ex:label
	Message: Value "hello"@fr should have the datatype rdf:dirLangString



#### 05.01.3.1.4 One value per language: `sh:uniqueLang`

With `sh:uniqueLang true` on a property shape, no two value nodes may use the same language tag. For direction-tagged strings (`rdf:dirLangString`, such as `"hello"@en--ltr`) SHACL 1.2 includes the base direction in the comparison: `"hello"@en--ltr` and `"salut"@en--rtl` are different, as are `"hello"@en--ltr` and `"hi"@en` (no direction). Only two values with the same language and the same direction (or both without a direction) violate the constraint.

Below, `ex:Alice` has two values for `ex:greeting` that differ in direction (conforms), and `ex:Bob` has two with the same tag and direction (does not conform).

In [7]:
shapes = StarLayerGraph()
shapes.parse(format="turtle", data="""
@prefix ex: <http://example.org/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person ;
  sh:property [ sh:path ex:greeting ; sh:uniqueLang true ] .
""")

# the processor is created from the shapes graph, then used to validate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)

good = StarLayerGraph()
good.parse(format="turtle", data="""
@prefix ex: <http://example.org/> .
ex:Alice a ex:Person ; ex:greeting "hello"@en--ltr, "hi"@en--rtl, "hey"@en .
""")
bad = StarLayerGraph()
bad.parse(format="turtle", data="""
@prefix ex: <http://example.org/> .
ex:Bob a ex:Person ; ex:greeting "hello"@en--ltr, "hi"@en--ltr .
""")
print(myProcessor.validate(data_graph=good).conforms)
result = myProcessor.validate(data_graph=bad)
print(result.conforms)
print(result.report_text)

True
False
Validation Report
Conforms: False
Results (1):
Constraint Violation in sh:UniqueLangConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:path ex:greeting ; sh:uniqueLang true ]
	Focus Node: ex:Bob
	Result Path: ex:greeting
	Message: Allows at most 1 value for ex:greeting with the language tag en--ltr, found 2



### 05.01.3.2 New constraint components

SHACL 1.2 introduces a variety of new constraint components.

#### 05.01.3.2.1 No line breaks in a value: `sh:singleLine`

`sh:singleLine true` says that no value node may contain a line break. A literal violates it if its lexical form contains a line feed, vertical tab, form feed or carriage return; a value that is not a literal is not checked. 

NOTE: In the current draft `sh:singleLine` is marked "at risk" pending a working group decision on convenience features like it, so it may change or move.

Below, `ex:alice`'s name is on one line (conforms), and `ex:bob`'s name contains a line break (does not conform).

In [8]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ; sh:singleLine true ] .
""", format="turtle")

good = StarLayerGraph()
good.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:name "Alice Smith" .
""", format="turtle")

bad = StarLayerGraph()
bad.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:bob a ex:Person ; ex:name "Bob\\nJones" .
""", format="turtle")

# the processor is created from the shapes graph, then used to validate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)
print("one line:", myProcessor.validate(data_graph=good).conforms)
result = myProcessor.validate(data_graph=bad)
print("with a line break:", result.conforms)
print(result.report_text)

one line: True
with a line break: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in sh:SingleLineConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:path ex:name ; sh:singleLine true ]
	Focus Node: ex:bob
	Value Node: "Bob
Jones"
	Result Path: ex:name
	Message: Value "Bob\nJones" should not contain a line break



#### 05.01.3.2.2 At least one value matches: `sh:someValue`

`sh:someValue` takes a shape and requires that at least one value node conforms to it.  Other values may not conform. It is the existential counterpart of `sh:node`, which requires every value to conform to a shape. It is also shorthand for `sh:qualifiedValueShape` with `sh:qualifiedMinCount 1`. It is mostly used in a property shape.

Below, every `ex:Farm` must tend an animal that conforms to `ex:DuckShape`. `ex:greenAcres` tends a cow and a duck (conforms) and `ex:dryGulch` tends a cow and a horse (does not conform). The result names the focus node but no value node, because the failure is about the values as a whole.

In [9]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:FarmShape a sh:NodeShape ;
      sh:targetClass ex:Farm ;
      sh:property [ sh:path ex:tendsAnimal ; sh:someValue ex:DuckShape ] .
    ex:DuckShape a sh:NodeShape ;
      sh:class ex:Duck .
""", format="turtle")

good = StarLayerGraph()
good.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:greenAcres a ex:Farm ; ex:tendsAnimal ex:daisy, ex:donald .
    ex:daisy a ex:Cow .
    ex:donald a ex:Duck .
""", format="turtle")

bad = StarLayerGraph()
bad.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:dryGulch a ex:Farm ; ex:tendsAnimal ex:bessie, ex:clyde .
    ex:bessie a ex:Cow .
    ex:clyde a ex:Horse .
""", format="turtle")

# the processor is created from the shapes graph, then used to validate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)
print("one duck among the animals:", myProcessor.validate(data_graph=good).conforms)
result = myProcessor.validate(data_graph=bad)
print("no duck:", result.conforms)
print(result.report_text)

one duck among the animals: True
no duck: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in sh:SomeValueConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:path ex:tendsAnimal ; sh:someValue ex:DuckShape ]
	Focus Node: ex:dryGulch
	Result Path: ex:tendsAnimal
	Message: No value of ex:tendsAnimal conforms to ex:DuckShape



#### 05.01.3.2.3 Unique values across focus nodes: `sh:uniqueValuesFor`

Every other constraint looks at one focus node at a time. `sh:uniqueValuesFor` is a node-shape constraint that compares the shape's target nodes with each other. No two target nodes may have exactly the same values for the named property(ies). Several properties can be given as a list in which case the combination of values must be unique.

- Only the shape's own targets count, so a node outside the target that shares the values is not a clash.
- A node with no value for all of the listed properties is skipped. A node with a value for some of them is still compared, and a missing value matches another missing value.
- Literals match exactly, so `"4"^^xsd:integer` differs from `"04"^^xsd:byte`. 
- Combine the constraint with `sh:minCount 1` to ensure values are also required.

Below, `ex:p3` and `ex:p4` share both names (do not conform), and `ex:p5` and `ex:p6` share a first name and have no last name (do not conform). `ex:p1` and `ex:p2` differ in the last name (conform), `ex:p7` and `ex:p8` have no names and are skipped (conform), and `ex:visitor` which shares names with `ex:p1` is not checked because it is not a `ex:Person`.

In [10]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:uniqueValuesFor ( ex:firstName ex:lastName ) .
""", format="turtle")

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:p1 a ex:Person ; ex:firstName "Ann" ; ex:lastName "Lee" .
    ex:p2 a ex:Person ; ex:firstName "Ann" ; ex:lastName "Smith" .
    ex:p3 a ex:Person ; ex:firstName "Bob" ; ex:lastName "Jones" .
    ex:p4 a ex:Person ; ex:firstName "Bob" ; ex:lastName "Jones" .
    ex:p5 a ex:Person ; ex:firstName "Cy" .
    ex:p6 a ex:Person ; ex:firstName "Cy" .
    ex:p7 a ex:Person .
    ex:p8 a ex:Person .
    ex:visitor ex:firstName "Ann" ; ex:lastName "Lee" .
""", format="turtle")

# the processor is created from the shapes graph, then used to validate the data graph
# p3 and p4 share both names and p5 and p6 share a first name and have no last name, so they clash.
# p1 and p2 differ in the last name, p7 and p8 have no names (skipped), and the visitor is not a Person.
myProcessor = StarShaclProcessor(shacl_graph=shapes)
result = myProcessor.validate(data_graph=data)
print("conforms:", result.conforms)
print(result.report_text)


conforms: False
Validation Report
Conforms: False
Results (4):
Constraint Violation in sh:UniqueValuesForConstraintComponent:
	Severity: sh:Violation
	Source Shape: ex:PersonShape
	Focus Node: ex:p3
	Message: ex:p3 has the same values as ex:p4 for ex:firstName "Bob", ex:lastName "Jones"
Constraint Violation in sh:UniqueValuesForConstraintComponent:
	Severity: sh:Violation
	Source Shape: ex:PersonShape
	Focus Node: ex:p4
	Message: ex:p4 has the same values as ex:p3 for ex:firstName "Bob", ex:lastName "Jones"
Constraint Violation in sh:UniqueValuesForConstraintComponent:
	Severity: sh:Violation
	Source Shape: ex:PersonShape
	Focus Node: ex:p5
	Message: ex:p5 has the same values as ex:p6 for ex:firstName "Cy", ex:lastName (none)
Constraint Violation in sh:UniqueValuesForConstraintComponent:
	Severity: sh:Violation
	Source Shape: ex:PersonShape
	Focus Node: ex:p6
	Message: ex:p6 has the same values as ex:p5 for ex:firstName "Cy", ex:lastName (none)



### 05.01.3.3 Lists

SHACL 1.2 expands the ability of SHACL to work with RDF lists.  

#### 05.01.3.3.1 Constraints on the members of a list 

When the value of a property is an RDF list, SHACL 1.2 Core has four constraint components that can look at the list itself. 

- `sh:memberShape` requires every member of the list to conform to a node shape. 
- `sh:minListLength` and `sh:maxListLength` bound the number of list members. 
- `sh:uniqueMembers true` forbids the same member twice. 

Each one also requires the value to be a well-formed list: a value that is not a list is itself a violation.

Below, the speakers of a talk are an ordered list of one to three IRIs with no one listed twice. `ex:talk1` has two speakers (conforms), `ex:talk2` to `ex:talk5` each break a different rule (do not conform), and `ex:talk6` has a single speaker that is not written as a list (does not conform), so it fails all four list components.

In [11]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:TalkShape a sh:NodeShape ;
      sh:targetClass ex:Talk ;
      sh:property [
        sh:path ex:speakers ;
        sh:memberShape [ sh:nodeKind sh:IRI ] ;
        sh:minListLength 1 ;
        sh:maxListLength 3 ;
        sh:uniqueMembers true
      ] .
""", format="turtle")

# the processor is created from the shapes graph, then used to validate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)

good = StarLayerGraph()
good.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:talk1 a ex:Talk ; ex:speakers ( ex:alice ex:bob ) .
""", format="turtle")
print("conforms:", myProcessor.validate(data_graph=good).conforms)

bad = StarLayerGraph()
bad.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:talk2 a ex:Talk ; ex:speakers ( ex:alice "Bob" ) .
    ex:talk3 a ex:Talk ; ex:speakers () .
    ex:talk4 a ex:Talk ; ex:speakers ( ex:alice ex:bob ex:carol ex:dana ) .
    ex:talk5 a ex:Talk ; ex:speakers ( ex:alice ex:alice ) .
    ex:talk6 a ex:Talk ; ex:speakers ex:alice .   # not a list at all
""", format="turtle")
result = myProcessor.validate(data_graph=bad)
print("conforms:", result.conforms)
print(result.report_text)


conforms: True
conforms: False
Validation Report
Conforms: False
Results (8):
Constraint Violation in sh:MaxListLengthConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:maxListLength 3 ; sh:memberShape [ sh:nodeKind sh:IRI ] ; sh:minListLength 1 ; sh:path ex:speakers ; sh:uniqueMembers true ]
	Focus Node: ex:talk4
	Value Node: ( ex:alice ex:bob ex:carol ex:dana )
	Result Path: ex:speakers
	Message: The list of ex:speakers has 4 members, at most 3 are allowed
Constraint Violation in sh:MaxListLengthConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:maxListLength 3 ; sh:memberShape [ sh:nodeKind sh:IRI ] ; sh:minListLength 1 ; sh:path ex:speakers ; sh:uniqueMembers true ]
	Focus Node: ex:talk6
	Value Node: ex:alice
	Result Path: ex:speakers
	Message: The value ex:alice of ex:speakers should be a list
Constraint Violation in sh:MemberShapeConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:maxListLength 3 ; sh:memberShape [ sh:nodeKind sh:IRI ] ; s

#### 05.01.3.3.2 A list of choices for `sh:class` and `sh:datatype`

In SHACL 1.2 `sh:class` and `sh:datatype` can now take an RDF list instead of a single value. The list means a choice.  With `sh:class ( ex:Cat ex:Dog )` a node conforms if it is either a cat or a dog.  With `sh:datatype ( xsd:string xsd:integer )` a node conforms if its datatype is one of the listed datatypes.

A list is a choice (or). Separate `sh:class` triples on the same shape are a conjunction (and): each one must hold, so `sh:class ex:Dog, ex:Trained` requires a value to be an instance of both classes. By contrast, `sh:class ( ex:Dog ex:Trained )` requires the value to be a dog or trained, so a trained pig would conform. `sh:datatype` is different: a shape may have only one `sh:datatype` value, so SHACL does not permit more than one datatype constraint.

Below, a pet must be a cat or a dog, a tag must be a string or an integer, and a guard must be both a dog and trained. `ex:alice` meets all three (conforms), and `ex:bob` has a pet parrot, a boolean tag and a guard who is only a dog (does not conform).

In [12]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:OwnerShape a sh:NodeShape ;
      sh:targetClass ex:Owner ;
      sh:property [ sh:path ex:pet ; sh:class ( ex:Cat ex:Dog ) ] ;
      sh:property [ sh:path ex:guard ; sh:class ex:Dog, ex:Trained ] ;
      sh:property [ sh:path ex:tag ; sh:datatype ( xsd:string xsd:integer ) ] .
""", format="turtle")

# the processor is created from the shapes graph, then used to validate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)

good = StarLayerGraph()
good.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Owner ; ex:pet ex:tom, ex:rex ; ex:tag "vip", 7 ; ex:guard ex:rex .
    ex:tom a ex:Cat .
    ex:rex a ex:Dog, ex:Trained .
""", format="turtle")
print("conforms:", myProcessor.validate(data_graph=good).conforms)

bad = StarLayerGraph()
bad.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:bob a ex:Owner ; ex:pet ex:polly ; ex:tag true ; ex:guard ex:buddy .
    ex:polly a ex:Parrot .
    ex:buddy a ex:Dog .
""", format="turtle")
result = myProcessor.validate(data_graph=bad)
print("conforms:", result.conforms)
print(result.report_text)


conforms: True
conforms: False
Validation Report
Conforms: False
Results (3):
Constraint Violation in sh:ClassConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:class ( ex:Cat ex:Dog ) ; sh:path ex:pet ]
	Focus Node: ex:bob
	Value Node: ex:polly
	Result Path: ex:pet
	Message: Value ex:polly should be an instance of one of ex:Cat, ex:Dog
Constraint Violation in sh:ClassConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:class ex:Dog, ex:Trained ; sh:path ex:guard ]
	Focus Node: ex:bob
	Value Node: ex:buddy
	Result Path: ex:guard
	Message: Value ex:buddy should be an instance of ex:Trained
Constraint Violation in sh:DatatypeConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:datatype ( xsd:string xsd:integer ) ; sh:path ex:tag ]
	Focus Node: ex:bob
	Value Node: true
	Result Path: ex:tag
	Message: Value true should have one of the datatypes xsd:string, xsd:integer



### 05.01.3.4 Property path comparisons

In SHACL 1.2 the parameter of the comparison constraints has been expanded to include property paths, and `sh:subsetOf` has been added. A value node is compared with the nodes reached through that property path from the same focus node: the value nodes are those reached through `sh:path`, and the comparison nodes are those reached through the path of the constraint.

| Constraint | Requires |
|---|---|
| `sh:equals` | the value nodes and the comparison nodes are the same set |
| `sh:disjoint` | no node is in both sets |
| `sh:subsetOf` | every value node is one of the comparison nodes |
| `sh:lessThan` | every value node is less than every comparison node |
| `sh:lessThanOrEquals` | every value node is less than or equal to every comparison node |

There is no greater-than constraint. To require one value to be greater than another, put the lesser value on `sh:path` and the greater value on the path of the constraint.

Below, a task is compared with its project. The path `( [ sh:inversePath ex:hasTask ] ex:deadline )` goes back to the project and on to its deadline: a task's due date must not be later than that deadline, its team must be exactly the project's team, and each assignee must be a member of the project. `ex:design` meets all three (conforms), `ex:testing` is due after the deadline (does not conform), and `ex:docs` has a different team and an assignee who is not a member (does not conform).

In [13]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:TaskShape a sh:NodeShape ;
      sh:targetClass ex:Task ;
      sh:property [ sh:path ex:due ; sh:lessThanOrEquals ( [ sh:inversePath ex:hasTask ] ex:deadline ) ] ;
      sh:property [ sh:path ex:team ; sh:equals ( [ sh:inversePath ex:hasTask ] ex:team ) ] ;
      sh:property [ sh:path ex:assignee ; sh:subsetOf ( [ sh:inversePath ex:hasTask ] ex:member ) ] .
""", format="turtle")

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:launch ex:deadline "2027-03-01"^^xsd:date ; ex:team ex:alpha ; ex:member ex:ann, ex:bo ; ex:hasTask ex:design, ex:testing, ex:docs .
    ex:design a ex:Task ; ex:due "2027-02-01"^^xsd:date ; ex:team ex:alpha ; ex:assignee ex:ann .
    ex:testing a ex:Task ; ex:due "2027-03-15"^^xsd:date ; ex:team ex:alpha ; ex:assignee ex:ann, ex:bo .
    ex:docs a ex:Task ; ex:due "2027-02-20"^^xsd:date ; ex:team ex:beta ; ex:assignee ex:cy .
""", format="turtle")

# the processor is created from the shapes graph, then used to validate the data graph
# testing is due after the project deadline, and docs has a different team than its project and an assignee who is not a member
myProcessor = StarShaclProcessor(shacl_graph=shapes)
result = myProcessor.validate(data_graph=data)
print("conforms:", result.conforms)
print(result.report_text)


conforms: False
Validation Report
Conforms: False
Results (4):
Constraint Violation in sh:EqualsConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:equals ( [ sh:inversePath ex:hasTask ] ex:team ) ; sh:path ex:team ]
	Focus Node: ex:docs
	Value Node: ex:alpha
	Result Path: ex:team
	Message: Value ex:alpha for ^ex:hasTask/ex:team is not also a value of ex:team
Constraint Violation in sh:EqualsConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:equals ( [ sh:inversePath ex:hasTask ] ex:team ) ; sh:path ex:team ]
	Focus Node: ex:docs
	Value Node: ex:beta
	Result Path: ex:team
	Message: Value ex:beta for ex:team is not also a value of ^ex:hasTask/ex:team
Constraint Violation in sh:LessThanOrEqualsConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:lessThanOrEquals ( [ sh:inversePath ex:hasTask ] ex:deadline ) ; sh:path ex:due ]
	Focus Node: ex:testing
	Value Node: "2027-03-15"^^xsd:date
	Result Path: ex:due
	Message: Value "2027-03-15"^^xsd:date shoul

### 05.01.3.5 Values below a root class: `sh:rootClass`

`sh:rootClass` is used to check whether a value node is a named class itself or a subclass below it in the `rdfs:subClassOf` hierarchy of the data graph. A value that is not an IRI or is not a class will fail. This is distinct from `sh:class`, which asks whether the value node is an instance of a class.

Below, `ex:holds` describes a zoo by the kinds of animals it holds. `ex:holds` expects a value that is a class.   `ex:Animal` (conforms), `ex:Dog`, a subclass of `ex:Animal` (conforms), `ex:Plant`, which is not an animal class (does not conform), and `ex:happyTheHippo`, who is an instance of `ex:Animal`, but not a class (does not conform).

In [14]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:ZooShape a sh:NodeShape ;
      sh:targetClass ex:Zoo ;
      sh:property [ sh:path ex:holds ; sh:rootClass ex:Animal ] .
""", format="turtle")

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Mammal rdfs:subClassOf ex:Animal .
    ex:Dog    rdfs:subClassOf ex:Mammal .
    ex:Plant  rdfs:subClassOf ex:Organism .
    ex:happyTheHippo a ex:Animal .
    ex:cityZoo a ex:Zoo ; ex:holds ex:Dog, ex:Animal, ex:Plant, ex:happyTheHippo .
""", format="turtle")

# the processor is created from the shapes graph, then used to validate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)
result = myProcessor.validate(data_graph=data)
print("conforms:", result.conforms)
print(result.report_text)

conforms: False
Validation Report
Conforms: False
Results (2):
Constraint Violation in sh:RootClassConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:path ex:holds ; sh:rootClass ex:Animal ]
	Focus Node: ex:cityZoo
	Value Node: ex:Plant
	Result Path: ex:holds
	Message: Value ex:Plant should be ex:Animal or a subclass of it
Constraint Violation in sh:RootClassConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:path ex:holds ; sh:rootClass ex:Animal ]
	Focus Node: ex:cityZoo
	Value Node: ex:happyTheHippo
	Result Path: ex:holds
	Message: Value ex:happyTheHippo should be ex:Animal or a subclass of it



### 05.01.3.6 Targets

SHACL 1.2 has new ways to say which nodes a shape checks. 

The `StarShaclProcessor` provides the `target_nodes()` method to list the nodes a shape targets.  Target nodes are the nodes `validate()` checks. The `target_shapes()` method does the reverse: it gives the shapes that target a node, which are the shapes `validate()` checks the node against.

#### 05.01.3.6.1 Selecting targets with a shape: `sh:targetWhere`

`sh:targetWhere` selects focus nodes based on all nodes that conform to a given shape.

Below, `ex:VoterShape` targets the nodes that conform to `ex:AdultPersonShape` (an age of 18 or more) and includes the constraint `ex:voted true`. `ex:Bob` is an adult who voted (conforms) and `ex:Dave` is an adult who did not (does not conform). `ex:Alice` has no age and `ex:Carol` is a child, so neither is a target.

In [15]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:AdultPersonShape a sh:NodeShape ;
      sh:class ex:Person ;
      sh:property [ sh:path ex:age ; sh:minCount 1 ; sh:minInclusive 18 ] .
    ex:VoterShape a sh:NodeShape ;
      sh:targetWhere ex:AdultPersonShape ;
      sh:property [ sh:path ex:voted ; sh:hasValue true ] .
""", format="turtle")

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:Alice a ex:Person .
    ex:Bob a ex:Person ; ex:age 21 ; ex:voted true .
    ex:Carol a ex:Person ; ex:age 10 .
    ex:Dave a ex:Person ; ex:age 40 ; ex:voted false .
""", format="turtle")

# the processor is created from the shapes graph, then used to validate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)
result = myProcessor.validate(data_graph=data)

print("targets:", sorted(data.qname(n) for n in myProcessor.target_nodes(data_graph=data, shape_node=EX.VoterShape)))
print("the shapes of ex:Bob:", sorted(data.qname(shape) for shape in myProcessor.target_shapes(node=EX.Bob, data_graph=data)))
print("conforms:", result.conforms)
print("focus nodes in the report:", sorted(result.report_graph.qname(n) for n in result.report_graph.objects(None, SH.focusNode)))

targets: ['ex:Bob', 'ex:Dave']
the shapes of ex:Bob: ['ex:VoterShape']
conforms: False
focus nodes in the report: ['ex:Dave']


#### 05.01.3.6.2 Selecting a node by naming its shape: `sh:shape`

SHACL 1.2 allows a node itself to declare a shape that should be used for validation. The triple is in the **data graph** and not the **shapes graph**.  The shape itself needs no target declaration.

Below, `ex:ContractV1Shape` requires a signed contract and `ex:ContractV2Shape` adds a jurisdiction. `ex:contract1` and `ex:contract2` name the version of the shape they were written under in the data, so each is checked against its own shape (both conform).

In [16]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:ContractV1Shape a sh:NodeShape ;
      sh:property [ sh:path ex:signed ; sh:hasValue true ] .
    ex:ContractV2Shape a sh:NodeShape ;
      sh:property [ sh:path ex:signed ; sh:hasValue true ] ;
      sh:property [ sh:path ex:jurisdiction ; sh:minCount 1 ] .
""", format="turtle")

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:contract1 a ex:Contract ; sh:shape ex:ContractV1Shape ; ex:signed true .
    ex:contract2 a ex:Contract ; sh:shape ex:ContractV2Shape ; ex:signed true ; ex:jurisdiction ex:UK .
""", format="turtle")

# the processor is created from the shapes graph, then used to validate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)
result = myProcessor.validate(data_graph=data)

for shape in (EX.ContractV1Shape, EX.ContractV2Shape):
    print(data.qname(shape), "targets:", sorted(data.qname(n) for n in myProcessor.target_nodes(data_graph=data, shape_node=shape)))
print("conforms:", result.conforms)


ex:ContractV1Shape targets: ['ex:contract1']
ex:ContractV2Shape targets: ['ex:contract2']
conforms: True


### 05.01.3.7 Severity, messages and conformance

#### 05.01.3.7.1 New severity levels: `sh:Debug`, `sh:Trace`

SHACL 1.2 adds two severities in addition to `sh:Info`/`sh:Warning`. Unlike those two, a `sh:Debug`/`sh:Trace` result **never** blocks conformance — it's recorded in the report but has no effect on `result.conforms`. This is useful for constraints you want visibility into without failing validation.

Below, `ex:alice` has no age, which is a `sh:Debug` result: it is reported, but `ex:alice` still conforms.

In [17]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
""", format="turtle")
shapes = StarLayerGraph()

# a missing age is a debug check
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:age ; sh:minCount 1 ; sh:severity sh:Debug ] .
""", format="turtle")
# the processor is created from the shapes graph, then used to validate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)
result = myProcessor.validate(data_graph=data)
print("conforms despite missing ex:age:", result.conforms)  # True - sh:Debug never blocks
print(result.report_text)

conforms despite missing ex:age: True
Validation Report
Conforms: True
Results (1):
Validation Result in sh:MinCountConstraintComponent:
	Severity: sh:Debug
	Source Shape: [ sh:minCount 1 ; sh:path ex:age ; sh:severity sh:Debug ]
	Focus Node: ex:alice
	Result Path: ex:age
	Message: Needs at least 1 value for ex:age



#### 05.01.3.7.2 Fine-grained severity via reification: `{| sh:severity ... |}`

SHACL 1.2 allows a single constraint-value triple to carry its own `sh:severity`, `sh:message` or `sh:deactivated` override. The override is written in the shapes graph with RDF 1.2's inline annotation shorthand. This is finer-grained than a shape's own `sh:severity`. A shape can have an overall severity and message, which every constraint of the shape uses. An annotation replaces them for the one constraint it is attached to.

Below, `ex:AgeShape` is a warning with a message, and its `sh:datatype` constraint is annotated with `sh:Violation` and its own message. `ex:alice` has no age (does not conform, with the shape's severity and message), and `ex:bob`'s age is not an integer (does not conform, with the annotation's severity and message).

In [18]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:AgeShape a sh:PropertyShape ;
      sh:targetClass ex:Person ;
      sh:path ex:age ;
      sh:severity sh:Warning ;
      sh:message "The age should be recorded" ;
      sh:minCount 1 ;
      sh:datatype xsd:integer {| sh:severity sh:Violation ; sh:message "The age must be a whole number" |} .
""", format="turtle")

# alice fails the shape's own minCount, bob fails the annotated datatype constraint
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
    ex:bob a ex:Person ; ex:age "twenty" .
""", format="turtle")

# the processor is created from the shapes graph, then used to validate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)
result = myProcessor.validate(data_graph=data)
print("conforms:", result.conforms)
print(result.report_text)

# the same processor, with a different option for this call: alice's warning is accepted, bob's violation is not
result2 = myProcessor.validate(data_graph=data, allow_warnings=True)
print("conforms with allow_warnings=True:", result2.conforms)


conforms: False
Validation Report
Conforms: False
Results (2):
Constraint Violation in sh:DatatypeConstraintComponent:
	Severity: sh:Violation
	Source Shape: ex:AgeShape
	Focus Node: ex:bob
	Value Node: "twenty"
	Result Path: ex:age
	Message: The age must be a whole number
Validation Result in sh:MinCountConstraintComponent:
	Severity: sh:Warning
	Source Shape: ex:AgeShape
	Focus Node: ex:alice
	Result Path: ex:age
	Message: The age should be recorded

conforms with allow_warnings=True: False


#### 05.01.3.7.3 Which severities make a report fail: `sh:conformanceDisallows`

A validation report says which severities count against conformance. By default `sh:Violation`, `sh:Warning` and `sh:Info` result in non-conformance (conforms = false).  `sh:Debug` and `sh:Trace` never result in non-conformance. 

The `sh:conformanceDisallows` parameter allows modification of this default behavior. With `allow_infos=True`, `sh:Info` results do not result in non-conformance. With `allow_warnings=True`, neither `sh:Warning` nor `sh:Info` results do. All results are still in the report, and only `sh:conforms` and `sh:conformanceDisallows` change.

Below, `ex:alice` has one result of each severity. She does not conform by default or with `allow_infos=True`, and conforms with `allow_warnings=True`.

In [19]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:S a sh:NodeShape ;
      sh:targetNode ex:alice ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ; sh:severity sh:Info ] ;
      sh:property [ sh:path ex:email ; sh:minCount 1 ; sh:severity sh:Warning ] ;
      sh:property [ sh:path ex:nickname ; sh:minCount 1 ; sh:severity sh:Debug ] ;
      sh:property [ sh:path ex:phone ; sh:minCount 1 ; sh:severity sh:Trace ] .
""", format="turtle")

# alice has none of the four properties, so there is one result of each severity: a name (Info), an email (Warning),
# a nickname (Debug) and a phone (Trace)
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
""", format="turtle")

# the processor is created from the shapes graph, then used to validate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)

for label, options in [("default", {}), ("allow_infos", {"allow_infos": True}), ("allow_warnings", {"allow_warnings": True})]:
    result = myProcessor.validate(data_graph=data, **options)
    disallows = sorted(o.value.split("#")[1] for o in result.report_graph.objects(None, SH.conformanceDisallows))
    severities = sorted(o.value.split("#")[1] for o in result.report_graph.objects(None, SH.resultSeverity))
    print(f"{label:15} conforms={result.conforms!s:6} disallows={disallows} results={severities}")

default         conforms=False  disallows=['Info', 'Violation', 'Warning'] results=['Debug', 'Info', 'Trace', 'Warning']
allow_infos     conforms=False  disallows=['Violation', 'Warning'] results=['Debug', 'Info', 'Trace', 'Warning']
allow_warnings  conforms=True   disallows=['Violation'] results=['Debug', 'Info', 'Trace', 'Warning']


### 05.01.3.8 Constant value nodes: `sh:defaultValue` and `sh:values`

A property shape checks its constraints against its value nodes. In SHACL these are typically the nodes reached by `sh:path`. SHACL 1.2 allows additional values to be added with `sh:values`. SHACL 1.2 also provides `sh:defaultValue`, which supplies a value when the nodes reached by `sh:path` and those produced by `sh:values` are all empty.

In Core SHACL both properties produce constant values only (an IRI or a literal) and require a `sh:path` that is a plain property IRI. Computing more complex values from the data requires node expressions, see the [node expressions guide](05.04-shacl-node-expressions.ipynb). 

Both change only the value nodes of the property shape that declares them. Other property shapes, and comparisons such as `sh:lessThan`, read the data graph and do not see them.

#### 05.01.3.8.1 A value when none is stored: `sh:defaultValue`

Most value constraints pass when there is nothing to check. With `sh:defaultValue`, a property with no stored value is checked as if it had the default. A default never replaces a stored value.

Below, every student's `ex:score` must be at least 50 (`sh:minInclusive 50`). The shape is used twice, without a default and with `sh:defaultValue 0`. The output shows each student as conforming or with the constraint they failed. `ex:ann` has a score of 80 (conforms) and `ex:bob` has 30 (does not conform either way). `ex:cy` has no score. Without a default there is nothing to check, so cy conforms. With the default cy's missing score is checked as 0 (does not conform).

Note below the code reuses the Student shape by defining it as a function and passing the additional constraint to the function to modify the shape string. `%s` marks the place for the extra property, which is empty the first time and `sh:defaultValue 0 ;` the second. The Turtle then parses the modified string. 

In [20]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:ann a ex:Student ; ex:score 80 .
    ex:bob a ex:Student ; ex:score 30 .
    ex:cy  a ex:Student .
""", format="turtle")

def student_shape(extra=""):
    """A shapes graph that requires an ex:score of at least 50, with `extra` added to the property shape."""
    # plain Python string formatting, not SHACL or Turtle: %s is replaced by `extra` before the text is parsed
    shapes = StarLayerGraph()
    shapes.parse(data="""
        @prefix ex: <http://example.org/> .
        @prefix sh: <http://www.w3.org/ns/shacl#> .
        ex:StudentShape a sh:NodeShape ; sh:targetClass ex:Student ;
          sh:property [ sh:path ex:score ; %s sh:minInclusive 50 ] .
    """ % extra, format="turtle")
    return shapes

def verdicts(result, nodes):
    """Each node with "conforms", or the constraint components it failed."""
    report = result.report_graph
    failed = {}
    for r in report.objects(None, SH.result):
        failed.setdefault(report.qname(report.value(r, SH.focusNode)), []).append(report.qname(report.value(r, SH.sourceConstraintComponent)))
    return {node: ", ".join(sorted(failed[node])) if node in failed else "conforms" for node in nodes}

# student_shape() gives the shape without the extra, student_shape("sh:defaultValue 0 ;") gives it with a default
# each processor is created from its own shapes graph, then used to validate the same data graph
plainProcessor = StarShaclProcessor(shacl_graph=student_shape())
defaultProcessor = StarShaclProcessor(shacl_graph=student_shape("sh:defaultValue 0 ;"))
students = ["ex:ann", "ex:bob", "ex:cy"]
print("without a default:", verdicts(plainProcessor.validate(data_graph=data), students))
print("with a default:   ", verdicts(defaultProcessor.validate(data_graph=data), students))

without a default: {'ex:ann': 'conforms', 'ex:bob': 'sh:MinInclusiveConstraintComponent', 'ex:cy': 'conforms'}
with a default:    {'ex:ann': 'conforms', 'ex:bob': 'sh:MinInclusiveConstraintComponent', 'ex:cy': 'sh:MinInclusiveConstraintComponent'}


#### 05.01.3.8.2 Adding values: `sh:values`

`sh:values` adds its values to the stored ones for every focus node. It does not replace them, and the added values count like stored ones for every constraint of the property shape. Because `sh:values` has then produced a value, the set is not empty and a `sh:defaultValue` on the same property shape is not used.

Like the previous section, the shape is built from a Python string with `%s` where the extra property goes. Alice reviewed every document whether she is a reviewer or not. `sh:values ex:alice` adds Alice to the reviewers of every document. Every document must have Alice as a reviewer (`sh:hasValue ex:alice`) and at least three reviewers (`sh:minCount 3`). The shape is used twice, without `sh:values` and with `sh:values ex:alice`.

- `ex:doc1` has two reviewers, neither of them Alice. Without `sh:values` it does not conform, and with it Alice is added, giving three reviewers including her (conforms).
- `ex:doc2` has two reviewers, Alice and one other. Adding Alice again changes nothing, so it still has two (does not conform either way).
- `ex:doc3` has three reviewers including Alice (conforms either way). Adding Alice again changes nothing.
- `ex:doc4` has three reviewers, none of them Alice. Without `sh:values` it does not conform (no Alice), and with it, it has four (conforms).

In Core the values are constants. A value taken from another property needs a node expression, see the [node expressions guide](05.04-shacl-node-expressions.ipynb).

In [21]:
values_data = StarLayerGraph()
values_data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:doc1 a ex:Document ; ex:reviewer ex:bob, ex:cy .
    ex:doc2 a ex:Document ; ex:reviewer ex:alice, ex:bob .
    ex:doc3 a ex:Document ; ex:reviewer ex:alice, ex:bob, ex:cy .
    ex:doc4 a ex:Document ; ex:reviewer ex:bob, ex:cy, ex:dan .
""", format="turtle")

def document_shape(extra=""):
    """A shapes graph that requires at least three reviewers, one of them ex:alice, with `extra` added to the property shape."""
    # plain Python string formatting, not SHACL or Turtle: %s is replaced by `extra` before the text is parsed
    shapes = StarLayerGraph()
    shapes.parse(data="""
        @prefix ex: <http://example.org/> .
        @prefix sh: <http://www.w3.org/ns/shacl#> .
        ex:DocumentShape a sh:NodeShape ; sh:targetClass ex:Document ;
          sh:property [ sh:path ex:reviewer ; %s sh:hasValue ex:alice ; sh:minCount 3 ] .
    """ % extra, format="turtle")
    return shapes

# the processors are created from the shapes graphs, then used to validate the data graph
documents = ["ex:doc1", "ex:doc2", "ex:doc3", "ex:doc4"]
plainProcessor = StarShaclProcessor(shacl_graph=document_shape())
valuesProcessor = StarShaclProcessor(shacl_graph=document_shape("sh:values ex:alice ;"))
print("without sh:values:", verdicts(plainProcessor.validate(data_graph=values_data), documents))
print("with sh:values:   ", verdicts(valuesProcessor.validate(data_graph=values_data), documents))

without sh:values: 

{'ex:doc1': 'sh:HasValueConstraintComponent, sh:MinCountConstraintComponent', 'ex:doc2': 'sh:MinCountConstraintComponent', 'ex:doc3': 'conforms', 'ex:doc4': 'sh:HasValueConstraintComponent'}
with sh:values:    {'ex:doc1': 'conforms', 'ex:doc2': 'sh:MinCountConstraintComponent', 'ex:doc3': 'conforms', 'ex:doc4': 'conforms'}


## 05.01.4 Running validation and reading the result

`validate()` returns a `ValidationReport`, the whole outcome of the validation. 

- `conforms` says whether the data conforms; 

- `report_graph` is the SHACL validation report, a `StarLayerGraph` with a `sh:ValidationReport` node that links to one `sh:ValidationResult` for each result.  A `sh:ValidationResult` contains the relevant `sh:focusNode`, `sh:resultPath`, `sh:sourceConstraintComponent`, `sh:sourceShape`, `sh:resultSeverity`, `sh:value` and `sh:resultMessage`.

- `report_text` is the same report in human-readable text.

Below, `ex:alice` has no name (does not conform), and the validation is printed as `conforms` = false, then the report graph, then the report text.

In [22]:
from starlayer.graph import StarLayerGraph
from starlayer.shacl import SH, validate

shapes_r = StarLayerGraph()
shapes_r.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ; sh:datatype xsd:string ] .
""", format="turtle")
data_r = StarLayerGraph()
data_r.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
    ex:bob a ex:Person ; ex:name "Bob" .
""", format="turtle")

# the processor is created from the shapes graph, then used to validate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes_r)
result = myProcessor.validate(data_graph=data_r)
print("conforms:", result.conforms)

# the report as a graph
show(result.report_graph)

# the same report as text
print(result.report_text)


conforms: False
@prefix ex: <http://example.org/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .

_:b0 a sh:ValidationReport ;
    sh:conformanceDisallows sh:Info, sh:Violation, sh:Warning ;
    sh:conforms false ;
    sh:result [
        a sh:ValidationResult ;
        sh:focusNode ex:alice ;
        sh:resultMessage "Needs at least 1 value for ex:name" ;
        sh:resultPath ex:name ;
        sh:resultSeverity sh:Violation ;
        sh:sourceConstraintComponent sh:MinCountConstraintComponent ;
        sh:sourceShape []
    ] .

Validation Report
Conforms: False
Results (1):
Constraint Violation in sh:MinCountConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:datatype xsd:string ; sh:minCount 1 ; sh:path ex:name ]
	Focus Node: ex:alice
	Result Path: ex:name
	Message: Needs at least 1 value for ex:name



The rest of this section covers the options and checks around a run:

- **05.01.4.1 Each graph of a dataset**: `validate_each()`.
- **05.01.4.2 When the shapes are wrong**: `ShapeCompileError` and `MetaShaclError`, which are both `ShapeError`s.
- **05.01.4.3 Entailment during validation**: requesting an entailment regime, and its cache.
- **05.01.4.4 Naming the graphs in a report**: `sh:usedDataGraph` and `sh:usedShapesGraph` on the validation activity.
- **05.01.4.5 Recording the processor configuration**: `include_used_configuration=True`.

### 05.01.4.1 Each graph of a dataset

SHACL validates one data graph at a time. When `validate()` is used with a `StarLayerDataset` it works off the union of its graphs and individual graph names are ignored.  

The processor's `validate_each()` validates the graphs of a dataset one by one against the same shapes, parsing the shapes once, and returns `{graph name: ValidationReport}` (`None` is the default graph, when it holds triples). It takes a `StarLayerDataset` (or a `pyoxigraph.Store`) and nothing else: to validate a list or mapping of graphs, add each graph to a dataset under a name first (`dataset.graph(name).extend(graph)`), and to validate a single graph, use `validate()`.

Below, `ex:ann`, who has a name (conforms), is in one graph and `ex:cy`, who has none (does not conform), is in another. The union does not conform because of `ex:cy`, and `validate_each()` shows which graph that is.

In [23]:
from starlayer.graph import StarLayerDataset

people = StarLayerDataset(default_union=False)
people.parse(data="""
    @prefix ex: <http://example.org/> .
    GRAPH <urn:team:a> { ex:ann a ex:Person ; ex:name "Ann" . }
    GRAPH <urn:team:b> { ex:cy a ex:Person . }
""", format="trig")

# the processor is created from the shapes graph, then used to validate the dataset
myProcessor = StarShaclProcessor(shacl_graph=shapes_r)

# validate() works off the union of the graphs: one report for the whole dataset
union = myProcessor.validate(data_graph=people)
print("union of the graphs:", "conforms" if union.conforms else "does not conform")

# validate_each() validates each graph on its own: one report for each graph
for name, each in myProcessor.validate_each(people).items():
    print(name, "conforms" if each.conforms else "does not conform")


union of the graphs: does not conform
<urn:team:a> conforms
<urn:team:b> does not conform


### 05.01.4.2 When the shapes are wrong

A malformed shapes graph is usually found when the processor is created, not during a later validation. Errors fall into three groups:

- **Found when the processor is created.** A shape that cannot be read, such as a node shape with a `sh:path`, or a constraint with an unusable value, such as `sh:minCount "many"`, raises a `ShapeCompileError`, whether or not any data reaches the shape.
- **Found only when the shape is used.** A few mistakes in the shapes graph cannot be seen until the shape runs. The text of a SPARQL query, a malformed node expression or a malformed rule raises a `ShapeCompileError` on the first `validate()` or `apply_rules()`. A failure of the run itself is an `ExecutionError` (section 05.02.5).
- **Found only with `meta_shacl=True`.** The processor can also check the shapes graph against SHACL's own shapes graph (the shapes for shapes) when it is created, and raises a `MetaShaclError` if it does not conform. It finds shapes that are not valid SHACL but that the engine would still run, such as two `sh:datatype` values on one shape or a malformed `sh:languageIn` range. It also reports a term in the SHACL namespace that SHACL does not define, such as `sh:minCountt`.

`meta_shacl=True` is useful while you are developing shapes. Once the shapes are finished there is no need to check them against the shapes for shapes on every use. `ShapeCompileError` and `MetaShaclError` are both `ShapeError`s, so one `except ShapeError` covers them. An `ExecutionError` is not a `ShapeError`: section 05.02.5 lists the errors. Each message says which shape, property or rule it is about, for example `the property shape of http://example.org/S for http://example.org/name: sh:minCount needs exactly one non-negative xsd:integer, got ['many']`.

Below, a node shape with a `sh:path` raises a `ShapeCompileError` when the processor is created, and a property shape with two `sh:datatype` values raises a `MetaShaclError` only when `meta_shacl=True`.

In [24]:
from starlayer.shacl import MetaShaclError, ShapeCompileError

# a node shape with a sh:path is always an error, found when the processor is created
misplaced = StarLayerGraph()
misplaced.parse(data="@prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> . ex:S a sh:NodeShape ; sh:path ex:p .", format="turtle")
try:
    myProcessor = StarShaclProcessor(shacl_graph=misplaced)
except ShapeCompileError as error:
    print("ShapeCompileError:", error)

# a shape that is not valid SHACL is found only with meta_shacl=True: two sh:datatype values on one shape
two_datatypes = StarLayerGraph()
two_datatypes.parse(data="""@prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> . @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:S a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ; sh:datatype xsd:string, xsd:integer ] .""", format="turtle")
myProcessor = StarShaclProcessor(shacl_graph=two_datatypes)   # the engine accepts it
try:
    myProcessor = StarShaclProcessor(shacl_graph=two_datatypes, meta_shacl=True)
except MetaShaclError as error:
    print("MetaShaclError:", error)


ShapeCompileError: http://example.org/S: a sh:NodeShape cannot be the subject of sh:path
MetaShaclError: The shapes graph is not valid SHACL: it does not validate against the SHACL shapes for shapes (MetaSHACL). 1 problem:
- the property shape of ex:S for ex:name, sh:datatype: Allows at most 1 value, found 2


### 05.01.4.3 Entailment during validation

SHACL follows `rdfs:subClassOf` chains to determine the nodes a shape targets, and whether a value is of a given class. No special settings are required.  (To read `rdfs:subClassOf` triples from the shapes graph as well, see [05.03 SHACL shapes as classes](05.03-shapes-as-classes.ipynb).)

Other entailments, such as those from `rdfs:domain` or `rdfs:range`, or more complex entailments, are not used by SHACL without explicit instructions. The regimes StarLayer can apply, and which one to choose, are described in [section 5 of the inferencing guide](02e-graphs-inferencing.ipynb). An ontology that is kept apart from the data can be given to the processor as `ont_graph`, and it is combined with the data when an entailment regime is applied.

An entailment regime can be requested in one of three ways:
- as a parameter of the `StarShaclProcessor`, which binds it for every call,
- as a parameter of a call to `validate()`, which replaces the processor's regime for that call only (`inference=None` switches entailment off for the call),
- or as a property of the shapes graph, using `sh:entailment` (see the [SHACL profiling guide](05.08-shacl-profiling.ipynb)).

The entailed triples are computed once and retained as a copy of the data graph with the entailed triples added. Validating the same data again with the same regime does not run the closure again, unless the data graph is modified (even with a different processor). This is the same cache that SPARQL uses for the OWL regimes (see [SPARQL entailment](03c-sparql-entailment.ipynb)), so a regime computed for a query is reused by a validation of the same graph.

Three settings control how the OWL regimes are computed, as they do for a SPARQL query. `recompute` says when the entailed triples are computed again: `"onChange"` (the default), `"never"` or `"always"`. It has no effect on the `RDF` and `RDFS` regimes. For the `OWL-Direct` regime only, `engine` chooses the reasoner (`"hermit"`, the default, or `"rustdl"`) and `timeout` limits the time it may take. Giving `engine` or `timeout` with another regime raises a `ValueError`. They are set on the processor like any other setting ([05.02.2](05.02-starlayer-shacl-engine.ipynb)), and the [SPARQL entailment guide](03c-sparql-entailment.ipynb) covers them fully.

Below, `ex:alice` is an `ex:Employee` only by entailment, because `ex:worksAt` has the domain `ex:Employee`. Without entailment the employee shape has no targets (conforms), and with the RDFS regime, requested in the three ways, `ex:alice` is a target and has no name (does not conform). `inference=None` switches the regime off for one call (conforms).

In [25]:
from starlayer.graph import ENTAILMENT

staff_shapes = StarLayerGraph()
staff_shapes.parse(data="""
    @prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:EmployeeShape a sh:NodeShape ; sh:targetClass ex:Employee ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ] .
""", format="turtle")
staff = StarLayerGraph()
staff.parse(data="""
    @prefix ex: <http://example.org/> . @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:worksAt rdfs:domain ex:Employee .
    ex:alice ex:worksAt ex:Acme .
""", format="turtle")

# no entailment: nobody is an ex:Employee, so the shape has no targets
plain = StarShaclProcessor(shacl_graph=staff_shapes)
print("targets, no entailment:", sorted(staff.qname(n) for n in plain.target_nodes(data_graph=staff, shape_node=EX.EmployeeShape)))
print("no entailment                   conforms:", plain.validate(data_graph=staff).conforms)

# 1. the regime is bound when the processor is created
rdfs_processor = StarShaclProcessor(shacl_graph=staff_shapes, inference=ENTAILMENT.RDFS)
print("targets, with inference:", sorted(staff.qname(n) for n in rdfs_processor.target_nodes(data_graph=staff, shape_node=EX.EmployeeShape)))
print("1. processor inference=         conforms:", rdfs_processor.validate(data_graph=staff).conforms)

# 2. the regime is passed to a call, which replaces the processor's regime for that call only
print("2. validate(inference=)         conforms:", plain.validate(data_graph=staff, inference=ENTAILMENT.RDFS).conforms)
print("   inference=None switches it off:", rdfs_processor.validate(data_graph=staff, inference=None).conforms)

# 3. the shapes graph asks for the regime with sh:entailment, a property of the shapes graph itself
entailing_shapes = StarLayerGraph()
entailing_shapes.parse(data="""
    @prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:StaffShapes a sh:ShapesGraph ;
      sh:entailment <http://www.w3.org/ns/entailment/RDFS> .
    ex:EmployeeShape a sh:NodeShape ; sh:targetClass ex:Employee ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ] .
""", format="turtle")
shapes_processor = StarShaclProcessor(shacl_graph=entailing_shapes)
print("3. sh:entailment in the shapes  conforms:", shapes_processor.validate(data_graph=staff).conforms)


targets, no entailment: []
no entailment                   conforms: True
targets, with inference: ['ex:alice']
1. processor inference=         conforms: False
2. validate(inference=)         conforms: False
   inference=None switches it off: True
3. sh:entailment in the shapes  conforms: False


### 05.01.4.4 Naming the graphs in a report

A report can record which graphs it is about. StarLayer follows the SHACL 1.2 Profiling draft ("Persisting validation results with context"): the validation is described as a `sh:ValidationActivity` that generated the report and used the graphs, with `sh:usedDataGraph` and `sh:usedShapesGraph` on the activity. The activity also names the processor, as a `sh:ValidationAgent`. The name of a graph is recorded when it is known, and nothing is written for a graph that has no name. When no graph has a name and no configuration is requested (see .4.5), no activity is written at all. A name can be provided in several ways:

- **The graph's own identifier**, given when the graph is created (`StarLayerGraph(identifier="http://example.org/people-2026")`).
- **The keywords `data_graph_iri=` and `shapes_graph_iri=`** on the call. A keyword takes precedence over the graph's identifier.
- **A named graph of a dataset**: `validate_each()` over a `StarLayerDataset` obtains each graph's name.
- **A shapes graph module**: when `shapes_graph_resource=` selects a `sh:ShapesGraph` module of the shapes graph, the module's IRI is recorded as the activity's `sh:usedShapesGraph`. (See the [SHACL profiling guide](05.08-shacl-profiling.ipynb).)

When both names are known and the data conforms, the report also says so in the way the SHACL 1.2 Profiling draft describes: the data graph is linked to the shapes graph with `sh:conformsToShapesGraph`. The [SHACL profiling guide](05.08-shacl-profiling.ipynb) covers this in detail.

Below, both graphs are named with the keywords. The data conforms, so the report holds an activity with the two names, and the `sh:conformsToShapesGraph` link.

In [26]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:S a sh:NodeShape ;
      sh:targetNode ex:alice ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ] .
""", format="turtle")

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:name "Alice" .
""", format="turtle")

# the processor is created from the shapes graph, then used to validate the data graph, naming both graphs
myProcessor = StarShaclProcessor(shacl_graph=shapes)
result = myProcessor.validate(
    data_graph=data,
    data_graph_iri="http://example.org/people-2026",
    shapes_graph_iri="http://example.org/people-shapes",
)
show(result.report_graph)


@prefix ex: <http://example.org/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix starlayer: <https://github.com/hidden-graph/starlayer/tree/main/packages/starlayer/starlayer/> .

ex:people-2026 sh:conformsToShapesGraph ex:people-shapes .

starlayer:shacl a sh:ValidationAgent .

_:b0 a sh:ValidationActivity ;
    prov:generated [
        a sh:ValidationReport ;
        sh:conformanceDisallows sh:Info, sh:Violation, sh:Warning ;
        sh:conforms true
    ] ;
    prov:wasAssociatedWith starlayer:shacl ;
    sh:usedDataGraph ex:people-2026 ;
    sh:usedShapesGraph ex:people-shapes .



### 05.01.4.5 Recording the processor configuration

Pass `include_used_configuration=True` to record the options the processor used for the run in the report.

The options are the settings of the processor, which are set when the processor is created and can be overridden for a single call (see the [engine guide](05.02-starlayer-shacl-engine.ipynb)). The report records the value in effect for the call, including the defaults.

Whether warnings and infos count against conformance is already on the report as `sh:conformanceDisallows`. Recording the options lets a reader of the report, or a later run, see how a result was produced, for example whether RDFS entailment was applied (see section 05.01.4.3) or whether `rdfs:subClassOf` was read from the shapes graph (see [shapes as classes](05.03-shapes-as-classes.ipynb)).

Below, the processor is created with RDFS entailment. The data conforms, and the report lists the options in effect for the run, including the entailment regime, and is shown as a graph. Without `include_used_configuration=True`, no configuration is written.

In [27]:
from starlayer.graph import ENTAILMENT

# the processor is created from the shapes graph, with RDFS entailment, then used to validate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes, inference=ENTAILMENT.RDFS)
result = myProcessor.validate(data_graph=data, include_used_configuration=True)
show(result.report_graph)

@prefix ent: <http://www.w3.org/ns/entailment/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix slopt: <urn:starlayer:shacl-engine:option:> .
@prefix starlayer: <https://github.com/hidden-graph/starlayer/tree/main/packages/starlayer/starlayer/> .

starlayer:shacl a sh:ValidationAgent .

_:b0 a sh:ValidationActivity ;
    prov:generated [
        a sh:ValidationReport ;
        sh:conformanceDisallows sh:Info, sh:Violation, sh:Warning ;
        sh:conforms true
    ] ;
    prov:wasAssociatedWith starlayer:shacl ;
    sh:usedConfiguration [
        a sh:ProcessorConfiguration ;
        slopt:abortOnFirst false ;
        slopt:ignoreImports false ;
        slopt:inference ent:RDFS ;
        slopt:inplace false ;
        slopt:maxValidationDepth 500 ;
        slopt:metaShacl false ;
        slopt:subclassOfInShapesGraph false
    ] .

